In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
df = pd.read_csv('olist.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 109547 entries, 0 to 109546
Data columns (total 30 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   price                          109547 non-null  float64
 1   freight_value                  109547 non-null  float64
 2   product_category_name          109547 non-null  str    
 3   product_name_lenght            109547 non-null  float64
 4   product_description_lenght     109547 non-null  float64
 5   product_photos_qty             109547 non-null  float64
 6   product_weight_g               109547 non-null  float64
 7   product_length_cm              109547 non-null  float64
 8   product_height_cm              109547 non-null  float64
 9   product_width_cm               109547 non-null  float64
 10  product_category_name_english  109547 non-null  str    
 11  order_status                   109547 non-null  str    
 12  review_score                   109547 non

# Train-Test split

In [4]:
target_columns = ['review_score', 'score_class']
feature_columns = [column for column in df.columns if column not in target_columns]

excluded_columns = ['review_comment_title', 'review_comment_message']
feature_columns = [
    column for column in feature_columns if column not in excluded_columns
]

X = df[feature_columns].replace([float('inf'), -float('inf')], np.nan)
y_regression = df['review_score']
y_classification = df['score_class']

X_regression_train, X_regression_test, y_regression_train, y_regression_test = train_test_split(
    X, y_regression, test_size=0.2, random_state=42
)

X_classification_train, X_classification_test, y_classification_train, y_classification_test = train_test_split(
    X, y_classification, test_size=0.2, random_state=42, stratify=y_classification
)

# Handling class imbalance

`score_category` is `Low` when `review_score < 4` and `High` otherwise. The
classification split is stratified, and LogisticRegression uses balanced class
weights so the minority class is not ignored.

# Baseline models

## Regression

The regression target is the numeric `review_score`. DummyRegressor predicts
its training-set median and provides a simple sanity-check benchmark.

In [5]:
dummy_regressor = DummyRegressor(strategy='median')
dummy_regressor.fit(
    pd.DataFrame({'constant': 0}, index=y_regression_train.index),
    y_regression_train
)
regression_predictions = dummy_regressor.predict(
    pd.DataFrame({'constant': 0}, index=y_regression_test.index)
)

regression_baseline_metrics = {
    'MAE': mean_absolute_error(y_regression_test, regression_predictions),
    'RMSE': root_mean_squared_error(
        y_regression_test, regression_predictions
    ),
    'R2': r2_score(y_regression_test, regression_predictions),
}

pd.Series(regression_baseline_metrics, name='DummyRegressor')

MAE     0.962300
RMSE    1.685387
R2     -0.483687
Name: DummyRegressor, dtype: float64

In [6]:
numeric_features = X_classification_train.select_dtypes(include='number').columns
categorical_features = X_classification_train.select_dtypes(exclude='number').columns

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_transformer, numeric_features),
    ('categorical', categorical_transformer, categorical_features),
])

logistic_baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler(with_mean=False)),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000)),
])

## Classification

The classification target is `score_class`. The mode classifier predicts the
most frequent class and provides the majority-class sanity check.

In [7]:
mode_classifier = DummyClassifier(strategy='most_frequent')
mode_classifier.fit(X_classification_train, y_classification_train)
mode_predictions = mode_classifier.predict(X_classification_test)

mode_classification_metrics = {
    'Accuracy': accuracy_score(y_classification_test, mode_predictions),
    'Balanced accuracy': balanced_accuracy_score(
        y_classification_test, mode_predictions
    ),
}

print(pd.Series(mode_classification_metrics, name='ModeClassifier'))
print('\nClassification report:')
print(
    classification_report(
        y_classification_test,
        mode_predictions,
        zero_division=0
    )
 )

Accuracy             0.756413
Balanced accuracy    0.500000
Name: ModeClassifier, dtype: float64

Classification report:
              precision    recall  f1-score   support

        High       0.76      1.00      0.86     16573
         Low       0.00      0.00      0.00      5337

    accuracy                           0.76     21910
   macro avg       0.38      0.50      0.43     21910
weighted avg       0.57      0.76      0.65     21910



In [8]:
logistic_baseline.fit(X_classification_train, y_classification_train)
logistic_predictions = logistic_baseline.predict(X_classification_test)

logistic_classification_metrics = {
    'Accuracy': accuracy_score(y_classification_test, logistic_predictions),
    'Balanced accuracy': balanced_accuracy_score(
        y_classification_test, logistic_predictions
    ),
}

print(pd.Series(logistic_classification_metrics, name='LogisticRegression'))
print('\nClassification report:')
print(classification_report(y_classification_test, logistic_predictions))

Accuracy             0.701004
Balanced accuracy    0.643950
Name: LogisticRegression, dtype: float64

Classification report:
              precision    recall  f1-score   support

        High       0.83      0.76      0.79     16573
         Low       0.41      0.53      0.46      5337

    accuracy                           0.70     21910
   macro avg       0.62      0.64      0.63     21910
weighted avg       0.73      0.70      0.71     21910

